In [ ]:
import os
import requests
from typing import TypedDict, List, Dict
from langgraph.graph import StateGraph
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [ ]:
class RepoState(TypedDict):
    owner: str
    repo: str
    files: List[str]
    current_file: str
    file_content: str
    chunks: List[Dict]
    issues: List[Dict]
    karma_score: float

In [ ]:
def get_repo_files(state):
    owner = state["owner"]
    repo = state["repo"]

    url = f"https://api.github.com/repos/{owner}/{repo}/git/trees/main?recursive=1"
    res = requests.get(url).json()

    files = []
    for item in res.get("tree", []):
        if item["type"] == "blob" and item["path"].endswith((".py", ".js", ".ts", ".vue", ".java")):
            files.append(item["path"])
    return {"files": files}

In [ ]:
def select_file(state):
    if not state["files"]:
        return {"current_file": None}
    return {"current_file": state["files"].pop()}

In [ ]:
def fetch_file_content(state):
    path = state["current_file"]
    if path is None:
        return {"file_content": ""}
    url = f"https://raw.githubusercontent.com/{state['owner']}/{state['repo']}/main/{path}"
    content = requests.get(url).text
    return {"file_content": content}

In [ ]:
def chunk_code(state):
    lines = state["file_content"].split("\n")
    chunks = []
    for i in range(0, len(lines), 40):
        chunk = "\n".join(lines[i:i+40])
        chunks.append({"code": chunk, "start_line": i+1})
    return {"chunks": chunks}

In [ ]:
llm = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device=-1,  # CPU; change to 0 if using GPU
    max_length=1024,
    temperature=0.2,
    do_sample=False
)

In [ ]:
def scan_code(state):
    issues = state.get("issues", [])
    file = state["current_file"]

    for chunk in state["chunks"]:
        prompt = f"""
You are a security code reviewer.
Find vulnerabilities in the following code.
Return JSON with:
issue, severity, line_number

Code:
{chunk['code']}
"""
        output = llm(prompt)[0]["generated_text"]

        if "issue" in output.lower():
            issues.append({
                "file": file,
                "line": chunk["start_line"],
                "description": output
            })

    return {"issues": issues}

In [ ]:
def compute_repo_score(state):
    score = 100
    for issue in state["issues"]:
        desc = issue["description"].lower()
        if "critical" in desc:
            score -= 10
        elif "medium" in desc:
            score -= 5
        else:
            score -= 2
    return {"karma_score": max(score, 0)}

In [ ]:
graph = StateGraph(RepoState)

graph.add_node("get_repo_files", get_repo_files)
graph.add_node("select_file", select_file)
graph.add_node("fetch_file_content", fetch_file_content)
graph.add_node("chunk_code", chunk_code)
graph.add_node("scan_code", scan_code)
graph.add_node("compute_repo_score", compute_repo_score)

graph.set_entry_point("get_repo_files")

graph.add_edge("get_repo_files","select_file")
graph.add_edge("select_file","fetch_file_content")
graph.add_edge("fetch_file_content","chunk_code")
graph.add_edge("chunk_code","scan_code")
graph.add_edge("scan_code","select_file")
graph.add_edge("select_file","compute_repo_score")

In [ ]:
!pip install graphviz matplotlib networkx

In [ ]:
app = graph.compile()

result = app.invoke({
    "owner": "Vedant122003",
    "repo": "Screenshot-sorter-tool",
    "issues": []
})

print("Repo Karma Score:", result["karma_score"])
for issue in result["issues"]:
    print(issue)